In [1]:
# Setting up three graph theory
K4 = ThreeGraphTheory(4, edges=[[0, 1, 2], [1, 2, 3], [0, 2, 3], [0, 1, 3]])
# C5 = ThreeGraphTheory.p(5, edges=[[0, 1, 2], [1, 2, 3], [2, 3, 4], [3, 4, 0], [4, 0, 1]])
F5c = ThreeGraphTheory.p(5, edges=[[0,1,4], [0,2,3], [0,2,4], [0,3,4], [1,2,3], [1,2,4], [1,3,4]])
ThreeGraphTheory.exclude([K4, F5c])
TG = ThreeGraphTheory

# Setting up colored theory
TwoColorTheory = combine("2Colors", Color0, Color1, symmetries=NoSymmetry)
TwoColorTheory.exclude([TwoColorTheory(1), TwoColorTheory(1, C0=[0], C1=[0])])
CTG = combine("2ColorThreeGraphs", TG, TwoColorTheory)

# Testing the generation of structures
print("Number of structures without F5c, K4")
print("and size 5: ", len(TG.generate(5)))
print("and size 6: ", len(TG.generate(6)))

print("\nNumber of 2 colored structures without F5c, K4")
print("and size 5: ", len(CTG.generate(5)))
print("and size 6: ", len(CTG.generate(6)))
TG.printlevel(0)

Number of structures without F5c, K4
and size 5:  22
and size 6:  818

Number of 2 colored structures without F5c, K4
and size 5:  366
and size 6:  41648


In [2]:
a31 = TG.optimize(TG(3, edges=[[0, 1, 2]]), 6, exact=True, denom=2**20, file="prop_3_1_k4f5c")
print("The initial upper bound on the Turan density from Proposition 3.1 is {} ~= {}".format(a31, a31.n()))

The initial upper bound on the Turan density from Proposition 3.1 is 30511/65536 ~= 0.465560913085938


In [3]:
# Generate good partition
target_size = 6
all_graphs = TG.generate(target_size, TG(3, ftype=[0, 1, 2], edges=[[0, 1, 2]]))

edge = TG(3, edges=[[0, 1, 2]])
target_typed = 0
for ii, flag in enumerate(all_graphs):
    nfp = flag.not_ftype_points()
    if flag.subflag(points=nfp, ftype_points=[]) != edge:
        continue
    A = 0
    for xx in nfp:
        touch = len(flag.subflag(points=[xx]).blocks("edges")) - 1
        if touch not in [0, 1]:
            A += 1
    if A == 2:
        target_typed += flag
target_flag = target_typed.project()


# Minimum degree
degree = TG(3, ftype=[0], edges=[[0, 1, 2]])
positives = [degree - 0.4641]

# Optimize
ratio = TG.optimize(target_flag, target_size, maximize=False, 
                    positives=positives, exact=True, denom=2**20, 
                    file="prop_3_2_k4f5c"
                   )
a32 = ratio / a31

print("The max-cut ratio returned by Proposition 3.2 is at least {} ~= {}".format(a32, a32.n()))

/home/shuaige/sage/src/sage/algebras/combinatorial_theory.py:1439: UserWarning: Using generic algorithm for an inexact ring, which will probably give incorrect results due to numerical precision issues.
  min(X_ii_small.eigenvalues())


The max-cut ratio returned by Proposition 3.2 is at least 43099222842404160803560878960795552015959/109451433676840840832138301242637359506224 ~= 0.393774858807753


In [4]:
target_size = 6

# Edge with correct colors
C = CTG(3, edges=[[0, 1, 2]], C0=[0, 1], C1=[2])
# Missing edge with good colors
M = CTG(3, edges=[], C0=[0, 1], C1=[2])
# Pointed edge with correct colors, point from color 0
Cp0 = CTG(3, edges=[[0, 1, 2]], C0=[0, 1], C1=[2], ftype=[0])
# Pointed edge with correct colors, point from color 1
Cp1 = CTG(3, edges=[[0, 1, 2]], C0=[0, 1], C1=[2], ftype=[2])


# Edge with bad colors all in color 0
B000 = CTG(3, edges=[[0, 1, 2]], C0=[0, 1, 2], C1=[])
# Edge with bad colors, looking the wrong way
B011 = CTG(3, edges=[[0, 1, 2]], C0=[0], C1=[1, 2])
# Pointed edge with bad colors, point from moving 1 -> 0
Bp0 = CTG(3, edges=[[0, 1, 2]], C0=[0, 1, 2], C1=[], ftype=[0])
# Pointed edge with bad colors, point from moving 0 -> 1
Bp1 = CTG(3, edges=[[0, 1, 2]], C0=[0], C1=[1, 2], ftype=[2])

# Degree equality conditions for various type pairs
degeq_00 = CTG.p(4, ftype=[0, 1], edges=[[0, 2, 3]], edges_m=[[1, 2, 3]], C0=[0, 1]) - \
           CTG.p(4, ftype=[1, 0], edges=[[0, 2, 3]], edges_m=[[1, 2, 3]], C0=[0, 1])
degeq_01 = CTG.p(4, ftype=[0, 1], edges=[[0, 2, 3]], edges_m=[[1, 2, 3]], C0=[0], C1=[1]) - \
           CTG.p(4, ftype=[0, 1], edges=[[1, 2, 3]], edges_m=[[0, 2, 3]], C0=[0], C1=[1])
degeq_11 = CTG.p(4, ftype=[0, 1], edges=[[0, 2, 3]], edges_m=[[1, 2, 3]], C1=[0, 1]) - \
           CTG.p(4, ftype=[1, 0], edges=[[0, 2, 3]], edges_m=[[1, 2, 3]], C1=[0, 1])

# Assumptions used in the proof
assums = [
    Cp0 - Bp0, Cp1 - Bp1, # Local optimality
    degeq_00, -degeq_00, degeq_01, -degeq_01, degeq_11, -degeq_11, # Degree regularity
    C - a32 # Best fit assumption
]

# Close rational approximation for gamma: 2521/5432
gamma = continued_fraction(2*sqrt(3) - 3).convergent(7)
# Construction
symbolic_constr = CTG.blowup_construction(target_size, ["X0", "X1"], edges=[[0, 0, 1]], C0=[[0]], C1=[[1]]).set_sum()
ders = symbolic_constr.derivatives([gamma])

/home/shuaige/sage/src/sage/algebras/combinatorial_theory.py:2920: UserWarning: The pattern is initialized with optional relations inside 
            the ftype. Those relations will be missing in the resulting pattern.
  return Pattern(self, n, ftype_points, **blocks)


In [5]:
CTG.optimize(B000 + B011 - (1 - 1/4000)*M, target_size, positives=assums, exact=True, maxiter = 300, 
              construction=ders, file="prop_3_3_k4f5c", denom=1024*225, slack_threshold=1e-6, kernel_denom=2**20
             )

Base flags generated, their number is 41648
The relevant ftypes are constructed, their number is 33
Block sizes before symmetric/asymmetric change is applied: [37, 37, 37, 128, 128, 128, 128, 128, 112, 112, 112, 112, 112, 112, 112, 112, 98, 98, 98, 98, 98, 98, 98, 98, 98, 78, 78, 78, 78, 78, 78, 78, 78]


Done with mult table for Ftype on 4 points with edges=(013 023 123), C0=(0 1 2 3), C1=(): : 33it [00:28,  1.16it/s]


Tables finished


Done with positivity constraint 8: 100%|█████████| 9/9 [28:53<00:00, 192.64s/it]


Constraints finished
Adjusting table with kernels from construction
Running SDP after kernel correction. Used block sizes are [26, 10, 36, 26, 10, 21, 106, 39, 88, 56, 72, 40, 88, 21, 106, 32, 80, 68, 44, 32, 80, 68, 44, 68, 44, 32, 80, 68, 44, 32, 80, 42, 56, 58, 40, 62, 36, 98, 42, 56, 41, 56, 62, 36, 58, 40, 42, 56, 24, 54, 24, 54, 48, 30, 48, 30, 48, 30, 48, 30, 23, 54, 24, 54, -41648, -304]
CSDP 6.2.0
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 1.04e-02 Pobj: -3.8394795e+00 Ad: 8.67e-03 Dobj:  1.1179888e+02 
Iter:  2 Ap: 1.66e-02 Pobj: -1.0300645e+01 Ad: 2.47e-02 Dobj:  1.1956427e+02 
Iter:  3 Ap: 4.64e-02 Pobj: -2.9864706e+01 Ad: 4.67e-02 Dobj:  1.3865806e+02 
Iter:  4 Ap: 9.00e-01 Pobj: -3.0562506e+02 Ad: 1.43e-01 Dobj:  1.1370225e+02 
Iter:  5 Ap: 1.00e+00 Pobj: -3.8599212e+02 Ad: 7.08e-01 Dobj:  3.5084140e+01 
Iter:  6 Ap: 1.00e+00 Pobj: -4.2759783e+02 Ad: 8.55e-01 Dobj:  5.0763085e+00 
Iter:  7 Ap: 1.00e+00 Pobj: -4.6713619e+02 A

100%|██████████████████████████████████████| 33/33 [42:01:41<00:00, 4584.90s/it]


This took 151306.797205925s
Final rounded bound is 0


0

In [2]:
TG.verify("prop_3_1_k4f5c",TG(3, edges=[[0, 1, 2]]), 6)

The solution is valid, it proves the bound 30511/65536


30511/65536

In [4]:
# Generate good partition
target_size = 6
all_graphs = TG.generate(target_size, TG(3, ftype=[0, 1, 2], edges=[[0, 1, 2]]))

edge = TG(3, edges=[[0, 1, 2]])
target_typed = 0
for ii, flag in enumerate(all_graphs):
    nfp = flag.not_ftype_points()
    if flag.subflag(points=nfp, ftype_points=[]) != edge:
        continue
    A = 0
    for xx in nfp:
        touch = len(flag.subflag(points=[xx]).blocks("edges")) - 1
        if touch not in [0, 1]:
            A += 1
    if A == 2:
        target_typed += flag
target_flag = target_typed.project()


# Minimum degree
degree = TG(3, ftype=[0], edges=[[0, 1, 2]])
positives = [degree - 0.4641]

TG.verify("prop_3_2_k4f5c",target_flag, target_size, maximize=False, positives=positives)

The solution is valid, it proves the bound 0.183326182816823


0.183326182816823